# Can we predict whether a student passes the closing quiz?
Target: `passed_quiz`. Follows `checklist.md`.

**Design decisions (made before looking at any model score)**
- `student_key` appears in many rows (~7 sessions each) and the data has no session date/order, so there is no time axis to split on. The test set therefore holds **whole students**; CV inside the training rows is also grouped by student.
- `student_key` is an identifier: used only as the grouping variable, never as a feature.
- Prediction time = just before the closing quiz is taken. `session_minutes` and `hints_used` are recorded during the session, so they are available then; `prior_score` is a placement test taken earlier. If the prediction were needed at session *start*, the two in-session features would not be usable - noted as a limitation.
- Preprocessing (scaling, one-hot) lives inside sklearn Pipelines so it is fit within each CV fold only.

In [ ]:
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, log_loss
SEED=0
df=pd.read_csv('data.csv')
print(df.shape, 'students:', df.student_key.nunique(), 'pass rate: %.3f'%df.passed_quiz.mean())
df.describe().T

## 1. Set aside the test set (whole students, 20%)

In [ ]:
gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=SEED)
tr_idx,te_idx=next(gss.split(df,groups=df.student_key))
train,test=df.iloc[tr_idx].reset_index(drop=True),df.iloc[te_idx].reset_index(drop=True)
assert not set(train.student_key)&set(test.student_key)
print(len(train),len(test),train.student_key.nunique(),test.student_key.nunique())
# test is not touched again until section 5
y=train.passed_quiz.values; groups=train.student_key.values

## 2. Candidate models (named before any scoring)
| name | features | model |
|---|---|---|
| `baseline` | none | predicts training pass rate |
| `logit_session` | prior_score, session_minutes, hints_used | logistic regression (C in {0.1, 1, 10}) |
| `logit_all` | the above + enrol_year + one-hot school_code | logistic regression (C in {0.1, 1, 10}) |
| `hgb_session` | prior_score, session_minutes, hints_used | gradient boosting (max_depth in {2, 3}, lr 0.05) |
| `hgb_all` | the above + enrol_year, school_code (numeric) | gradient boosting (max_depth in {2, 3}, lr 0.05) |

Selection metric: mean CV ROC AUC (out-of-fold). Decision threshold: the one maximising balanced accuracy on out-of-fold predictions of the chosen model. Near-ties (within 1 SE of the best) go to the simpler model.

In [ ]:
S=['prior_score','session_minutes','hints_used']
A=S+['enrol_year','school_code']
def logit(cols,C_,cat=False):
    num=[c for c in cols if c!='school_code']
    tf=[('n',StandardScaler(),num)]
    if cat: tf.append(('c',OneHotEncoder(handle_unknown='ignore'),['school_code']))
    return Pipeline([('p',ColumnTransformer(tf)),('m',LogisticRegression(C=C_,max_iter=2000))])
def hgb(d): return Pipeline([('m',HistGradientBoostingClassifier(max_depth=d,learning_rate=0.05,max_iter=200,random_state=SEED))])
cands={'baseline':(['prior_score'],Pipeline([('m',DummyClassifier(strategy='prior'))]))}
for c in [0.1,1,10]: cands[f'logit_session C={c}']=(S,logit(S,c))
for c in [0.1,1,10]: cands[f'logit_all C={c}']=(A,logit(A,c,cat=True))
for d in [2,3]: cands[f'hgb_session d={d}']=(S,hgb(d))
for d in [2,3]: cands[f'hgb_all d={d}']=(A,hgb(d))
list(cands)

## 3. Grouped cross-validation on the training rows only

In [ ]:
cv=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
folds=list(cv.split(train,y,groups))
res,oof={},{}
for name,(cols,pipe) in cands.items():
    p=np.zeros(len(train)); aucs=[]
    for a,b in folds:
        m=clone(pipe).fit(train.loc[a,cols],y[a])
        p[b]=m.predict_proba(train.loc[b,cols])[:,1]; aucs.append(roc_auc_score(y[b],p[b]))
    oof[name]=p; res[name]=(np.mean(aucs),np.std(aucs,ddof=1)/np.sqrt(len(aucs)),log_loss(y,p))
cvtab=pd.DataFrame(res,index=['auc','se','logloss']).T.sort_values('auc',ascending=False); cvtab

## 4. Choose the model and threshold on CV only

In [ ]:
best=cvtab.auc.idxmax(); cutoff=cvtab.auc[best]-cvtab.se[best]
chosen=next(n for n in cands if cvtab.auc[n]>=cutoff)   # cands is ordered simple -> complex
ts=np.linspace(0.2,0.8,61)
bal=[balanced_accuracy_score(y,oof[chosen]>=t) for t in ts]
threshold=float(ts[int(np.argmax(bal))])
print('best by AUC:',best,'| chosen (1-SE rule):',chosen,'| threshold: %.2f'%threshold)
print('CV AUC %.3f +/- %.3f ; CV balanced acc at threshold %.3f'%(cvtab.auc[chosen],cvtab.se[chosen],max(bal)))
print('baseline CV AUC %.3f'%cvtab.auc['baseline'])

**Frozen choice** - the model, feature set, hyperparameters and threshold printed above are fixed from this point. The test set is scored once, next, with no further changes.

## 5. Refit on all training rows and score the test set once

In [ ]:
cols,pipe=cands[chosen]
final=clone(pipe).fit(train[cols],y)
pt=final.predict_proba(test[cols])[:,1]; yt=test.passed_quiz.values
base=clone(cands['baseline'][1]).fit(train[['prior_score']],y).predict_proba(test[['prior_score']])[:,1]
def rep(p): return dict(auc=roc_auc_score(yt,p),logloss=log_loss(yt,p),bal_acc=balanced_accuracy_score(yt,p>=threshold),acc=accuracy_score(yt,p>=threshold))
out=pd.DataFrame({chosen:rep(pt),'baseline':rep(base)}).T
# student-level bootstrap CI for the headline AUC
rng=np.random.default_rng(SEED); ids=test.student_key.values; u=np.unique(ids)
idx_by={s:np.where(ids==s)[0] for s in u}; bs=[]
for _ in range(500):
    ii=np.concatenate([idx_by[s] for s in rng.choice(u,len(u))])
    if len(set(yt[ii]))==2: bs.append(roc_auc_score(yt[ii],pt[ii]))
print('Headline test AUC %.3f (95%% student-bootstrap CI %.3f-%.3f)'%(out.auc[chosen],*np.percentile(bs,[2.5,97.5])))
out

## 6. Conclusion and caveats
The headline is the single test-set score above; compare it to 0.5 / the baseline and to the CV estimate (a large gap would indicate selection optimism).
- Split is by student only because there is no date column; applying the model to future sessions of already-seen students is a different setting than the one evaluated (new students).
- `session_minutes` and `hints_used` are only known once the session is underway; the model is not usable at session start.
- `prior_score` is stored per session record and varies within a student (and has a few values >100); its provenance should be confirmed, since if it is refreshed after the quiz it would leak.